In [ ]:
# Cell 1: Import dependencies and resolve local config paths
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import yaml
from scipy.io import loadmat
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import load_model

notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text())
    dcfg = cfg.get('datasets', {}).get('deepradar2022', {})
    x_path = Path(dcfg.get('x_test', '/scratch/rameyjm7/datasets/DeepRadar2022/X_test.mat'))
    y_path = Path(dcfg.get('y_test', '/scratch/rameyjm7/datasets/DeepRadar2022/Y_test.mat'))
    lbl_path = Path(dcfg.get('lbl_test', '/scratch/rameyjm7/datasets/DeepRadar2022/lbl_test.mat'))
else:
    x_path = Path('/scratch/rameyjm7/datasets/DeepRadar2022/X_test.mat')
    y_path = Path('/scratch/rameyjm7/datasets/DeepRadar2022/Y_test.mat')
    lbl_path = Path('/scratch/rameyjm7/datasets/DeepRadar2022/lbl_test.mat')

model_path = project_root / 'models' / 'deepradar2022' / 'deepradar2022_cnn_bilstm_final.keras'
print('DeepRadar2022 X:', x_path)
print('DeepRadar2022 Y:', y_path)
print('DeepRadar2022 lbl:', lbl_path)
print('DeepRadar2022 model:', model_path)
assert x_path.exists() and y_path.exists() and lbl_path.exists(), 'Missing DeepRadar2022 files.'
assert model_path.exists(), f'Missing model: {model_path}'

In [ ]:
# Cell 2: Load test split and build model input features
with h5py.File(x_path, 'r') as h5:
    x_raw = np.asarray(h5['X_test'][:], dtype=np.float32)

x_iq = np.transpose(x_raw, (2, 1, 0))

y_mat = loadmat(y_path)
lbl_mat = loadmat(lbl_path)
y_key = next(k for k in y_mat.keys() if not k.startswith('__'))
y_onehot = y_mat[y_key]

y_true = np.argmax(y_onehot, axis=1).astype(np.int64)

envelope = np.sqrt(np.sum(np.square(x_iq), axis=2, keepdims=True))
X_eval = np.concatenate([x_iq, envelope], axis=2).astype(np.float32)

print('X_eval shape:', X_eval.shape)
print('class count:', y_onehot.shape[1])

In [ ]:
# Cell 3: Run model inference and print metrics
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_eval, verbose=0), axis=1)

acc = accuracy_score(y_true, y_pred)
print(f'DeepRadar2022 evaluation accuracy: {acc:.4f}')
print(classification_report(y_true, y_pred, zero_division=0))

In [ ]:
# Cell 4: Plot confusion matrix for DeepRadar2022
n_classes = int(max(y_true.max(), y_pred.max()) + 1)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues')
plt.title('DeepRadar2022 Confusion Matrix (class index view)')
plt.xlabel('Predicted class index')
plt.ylabel('True class index')
plt.tight_layout()
plt.show()